In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install torch torchvision

In [5]:
import os
import torch
import torchvision.models as models
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import json

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()
print(f"Training on: {device}")

soil_path = '/content/drive/MyDrive/AgriAI/datasets/soil_types'
soil_dataset = datasets.ImageFolder(soil_path, transform=transform)

soil_train_size = int(0.8 * len(soil_dataset))
soil_test_size = len(soil_dataset) - soil_train_size
soil_train, soil_test = torch.utils.data.random_split(
    soil_dataset, [soil_train_size, soil_test_size]
)
soil_train_loader = DataLoader(soil_train, batch_size=32,
                                shuffle=True, num_workers=2)

print(f"Total soil images: {len(soil_dataset)}")
print(f"Soil classes: {soil_dataset.classes}")

soil_model = models.resnet50(pretrained=True)
soil_model.fc = nn.Linear(
    soil_model.fc.in_features,
    len(soil_dataset.classes)
)
soil_model = soil_model.to(device)
soil_optimizer = torch.optim.Adam(soil_model.parameters(), lr=0.001)

print("Starting soil model training...")

for epoch in range(5):
    soil_model.train()
    running_loss = 0
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(soil_train_loader):
        images, labels = images.to(device), labels.to(device)
        soil_optimizer.zero_grad()
        outputs = soil_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        soil_optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if (i + 1) % 10 == 0:
            print(f"  Epoch {epoch+1} | Batch {i+1} | "
                  f"Acc: {100*correct/total:.2f}%")

    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/5] Complete — "
          f"Accuracy: {epoch_acc:.2f}%")

print("Soil model training complete!")

Training on: cpu
Total soil images: 1189
Soil classes: ['Alluvial_Soil', 'Arid_Soil', 'Black_Soil', 'Laterite_Soil', 'Mountain_Soil', 'Red_Soil', 'Yellow_Soil']


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s]


Starting soil model training...
  Epoch 1 | Batch 10 | Acc: 63.44%
  Epoch 1 | Batch 20 | Acc: 63.59%
  Epoch 1 | Batch 30 | Acc: 67.51%
Epoch [1/5] Complete — Accuracy: 67.51%
  Epoch 2 | Batch 10 | Acc: 73.12%
  Epoch 2 | Batch 20 | Acc: 73.59%
  Epoch 2 | Batch 30 | Acc: 74.03%
Epoch [2/5] Complete — Accuracy: 74.03%
  Epoch 3 | Batch 10 | Acc: 76.56%
  Epoch 3 | Batch 20 | Acc: 75.78%
  Epoch 3 | Batch 30 | Acc: 78.02%
Epoch [3/5] Complete — Accuracy: 78.02%
  Epoch 4 | Batch 10 | Acc: 77.81%
  Epoch 4 | Batch 20 | Acc: 81.25%
  Epoch 4 | Batch 30 | Acc: 81.49%
Epoch [4/5] Complete — Accuracy: 81.49%
  Epoch 5 | Batch 10 | Acc: 89.38%
  Epoch 5 | Batch 20 | Acc: 86.56%
  Epoch 5 | Batch 30 | Acc: 84.12%
Epoch [5/5] Complete — Accuracy: 84.12%
Soil model training complete!


In [6]:
save_path = '/content/drive/MyDrive/AgriAI/models/'
os.makedirs(save_path, exist_ok=True)

# Save model weights
torch.save(
    soil_model.state_dict(),
    save_path + 'soil_model.pth'
)

# Save class names
with open(save_path + 'soil_class_names.json', 'w') as f:
    json.dump(soil_dataset.classes, f)

print("Files saved to Google Drive:")
print(f"  soil_model.pth")
print(f"  soil_class_names.json")
print(f"\nSoil classes: {soil_dataset.classes}")
print("\nDONE! Go to Google Drive and download both files.")

Files saved to Google Drive:
  soil_model.pth
  soil_class_names.json

Soil classes: ['Alluvial_Soil', 'Arid_Soil', 'Black_Soil', 'Laterite_Soil', 'Mountain_Soil', 'Red_Soil', 'Yellow_Soil']

DONE! Go to Google Drive and download both files.
